# Viettel AI Race 2026 — Private Group B
This notebook trains four private scenes on two T4 GPUs with separate network ports. Add the existing `vai_nvs_kaggle_prod_v3.zip` dataset before running.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'CUDA is not available'
assert torch.cuda.device_count() >= 2, 'Group B requires GPU T4 x2'
print('GPU count:', torch.cuda.device_count())

In [ ]:
from pathlib import Path
import shutil

input_root = Path('/kaggle/input')
work_dir = Path('/kaggle/working/vai_nvs')
if work_dir.exists():
    shutil.rmtree(work_dir)

zip_candidates = list(input_root.rglob('vai_nvs_kaggle_prod_v3.zip'))
if zip_candidates:
    print('Extracting:', zip_candidates[0])
    shutil.unpack_archive(str(zip_candidates[0]), str(work_dir))
else:
    candidates = []
    for p in input_root.rglob('gaussian-splatting'):
        parent = p.parent
        if (parent / 'tools').exists() and (parent / 'work_undistorted').exists():
            candidates.append(parent)
    assert candidates, 'Could not find prod_v3 under /kaggle/input'
    shutil.copytree(candidates[0], work_dir)

assert (work_dir / 'gaussian-splatting').exists()
assert (work_dir / 'work_undistorted').exists()
print('Workspace ready:', work_dir)

In [ ]:
%cd /kaggle/working/vai_nvs

In [ ]:
!python -m pip install -U pip "setuptools<82" wheel ninja
!pip install tqdm plyfile opencv-python pillow scipy matplotlib

In [ ]:
import os
import torch
major, minor = torch.cuda.get_device_capability(0)
os.environ['TORCH_CUDA_ARCH_LIST'] = f'{major}.{minor}'
print('CUDA architecture:', os.environ['TORCH_CUDA_ARCH_LIST'])
!bash scripts/setup_cuda_extensions_linux.sh

In [ ]:
!python tools/inspect_undistorted_dataset.py --data_root work_undistorted

In [ ]:
from pathlib import Path
import subprocess
root = Path('/kaggle/working/vai_nvs/work_undistorted/phase1/private_set1')
for scene in sorted(root.iterdir()):
    if scene.is_dir():
        subprocess.run([
            'python', 'tools/prune_colmap_images_to_existing.py',
            '--scene_train_path', str(scene / 'train'),
        ], check=True)
print('Private pruning completed')

In [ ]:
from pathlib import Path
import shutil
runner_candidates = list(Path('/kaggle/input').rglob('train_private_group_b_2gpu.sh'))
assert runner_candidates, 'Add the Group B runner ZIP/dataset as a Kaggle input'
runner_target = Path('scripts/train_private_group_b_2gpu.sh')
shutil.copy2(runner_candidates[0], runner_target)
print('Using runner:', runner_candidates[0])
!bash scripts/train_private_group_b_2gpu.sh

In [ ]:
from pathlib import Path
scenes = ['HNI0366', 'HCM1439', 'HNI0437', 'HNI0265']
for scene in scenes:
    ply = Path(f'outputs/prod_private_{scene}/point_cloud/iteration_30000/point_cloud.ply')
    assert ply.exists(), f'Missing final model: {scene}'
    print(scene, 'OK', round(ply.stat().st_size / 1024**2, 1), 'MiB')
print('GROUP B VERIFIED')